# Step 1b (alternative): Download podcast audio with PytubeFix

Use this only if step 1a (yt-dlp) does not work. It downloads the same links from `data/podcasts.md` and saves WAV files named by video ID in `data/raw_audios/`.

- Needs `ffmpeg` on your PATH.
- Files that already exist are skipped.
- YouTube may block it with a "detected as a bot" error. That comes from YouTube, not from this code.

In [ ]:
import re
import subprocess
import tempfile
from pathlib import Path

from pytubefix import YouTube

# Find the project folder (works from any notebook folder)
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
)
podcasts_file = project_root / "data" / "podcasts.md"
output_dir = project_root / "data" / "raw_audios"
output_dir.mkdir(parents=True, exist_ok=True)

## 1. Read the links

In [ ]:
links = re.findall(r"https?://[^\s)>]+", podcasts_file.read_text(encoding="utf-8"))
print(f"Found {len(links)} links in {podcasts_file.relative_to(project_root)}")

## 2. Download

In [ ]:
def download_audio(link):
    video = YouTube(link)
    wav_file = output_dir / f"{video.video_id}.wav"
    if wav_file.exists():
        print(f"Already downloaded: {wav_file.name}")
        return

    stream = video.streams.get_audio_only()
    if stream is None:
        raise ValueError("No audio stream found")

    # Download to a temporary folder, convert to WAV, then move into place
    with tempfile.TemporaryDirectory(dir=output_dir) as temp_dir:
        source_file = stream.download(output_path=temp_dir, filename="audio")
        if not source_file:
            raise RuntimeError("Download did not produce an audio file")
        temp_wav = Path(temp_dir) / "audio.wav"
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", source_file, "-vn", "-acodec", "pcm_s16le", str(temp_wav)],
            check=True,
        )
        temp_wav.rename(wav_file)
    print(f"Saved: {wav_file.name}")


failed = []
for index, link in enumerate(links, start=1):
    print(f"({index}/{len(links)}) {link}")
    try:
        download_audio(link)
    except Exception as error:
        failed.append(link)
        print(f"Failed: {error}")

print(f"Finished: {len(links) - len(failed)} succeeded, {len(failed)} failed.")
print(f"Audio folder: {output_dir.relative_to(project_root)}")